# 1. Extração e Análise Exploratória Inicial

Nesta primeira etapa do pipeline de ETL, realizamos a leitura dos dados brutos a partir do arquivo CSV e executamos a inspeção inicial da estrutura da base de dados (`shape`, `info`, `describe`).

### Objetivos:
- Validar o carregamento dos dados com a codificação (`utf-8`) e separador (`.csv`) corretos.
- Identificar o número total de registros e colunas.
- Mapear a presença de valores nulos e verificar os tipos primitivos de dados lidos pelo Pandas.

In [ ]:
import pandas as pd 
import numpy as np 
import requests
import difflib

df_clientes = pd.read_csv('../data/clientes_distribuidora_vetor.csv', sep=';', encoding='utf-8') #enconding trabalha com caracteres especiais e sep trabalha com o separador do arquivo csv

display(df_clientes.head(14)) #deixa mais visualmente bonito a visualização do dataframe
display (df_clientes.shape) #mostra a quantidade de linhas e colunas do dataframe

display(df_clientes.info()) #mostra informações do dataframe, como quantidade de linhas, colunas, tipos de dados e valores nulos
display(df_clientes.describe()) #mostra estatísticas descritivas do dataframe, como média, mediana, desvio padrão, valores mínimos e máximos

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926-750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28.327.648/3503-05,+55 61 95386-2084,contato@comercialsão.com.br,Sul (Aguas Claras),72140-010,Brasília,DF,2023-10-07,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42.388.496/9653-28,+55 61 92489-1771,contato@armarinhossi.com.br,Taguatinga - DF,72140-010,Brasília,DF,28/02/2022,Camila Torres
3,CLI0004,Empório Vale Verde,69.784.801/8451-46,(61) 98433-1053,contato@empóriovalev.com.br,aguas claras,71929-000,Brasília,DF,"Dec 24, 2023",Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89.325.288/0957-01,61951273847,contato@merceariadoi.com.br,Taguatinga/DF,00000-000,Brasília,DF,08-01-2022,Diego Martins
5,CLI0006,Casa das Tintas Ipê,82.278.248/9638-34,+55 61 97118-8177,contato@casadastinta.com.br,Águas Claras,NaN,Brasília,DF,2021-08-17,Camila Torres
6,CLI0007,Distribuidora Popular Centro-Oeste,83.930.103/1051-83,61999788677,contato@distribuidor.com.br,TAGUATINGA,NaN,Brasília,DF,2025-04-16,Camila Torres
7,CLI0008,Bazar Estrela Dourada,31.165.667/0106-51,(61) 94139-4116,contato@bazarestrela.com.br,V. Pires,72006-400,Brasília,DF,"Aug 18, 2022",Juliana Reis
8,CLI0009,Supermercado Nova Aliança,10.801.326/7736-02,+55 61 91035-7396,contato@supermercado.com.br,Aguas claras,71926-750,Brasília,DF,2023-08-09,Juliana Reis
9,CLI0010,Papelaria Horizonte,34.309.805/0097-88,(61) 91931-9320,contato@papelariahor.com.br,Águas Claras/DF,71950-000,Brasília,DF,03/03/2021,Diego Martins


(47, 11)

<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   id_cliente            47 non-null     str  
 1   razao_social          47 non-null     str  
 2   cnpj                  47 non-null     str  
 3   telefone              47 non-null     str  
 4   email                 47 non-null     str  
 5   bairro_informado      46 non-null     str  
 6   cep                   34 non-null     str  
 7   cidade                47 non-null     str  
 8   uf                    47 non-null     str  
 9   data_cadastro         47 non-null     str  
 10  vendedor_responsavel  47 non-null     str  
dtypes: str(11)
memory usage: 4.2 KB


None

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
count,47,47,47,47,47,46,34,47,47,47,47
unique,47,47,47,47,44,25,15,1,1,47,5
top,CLI0001,Mercadinho Boa Vista,84.959.310/3413-16,61970855700,contato@distribuidor.com.br,Aguas claras,72140-010,Brasília,DF,2022-06-21,Camila Torres
freq,1,1,1,1,4,4,4,47,47,1,14


### 2. Transformação e Limpeza de Dados (ETL)

Nesta etapa do pipeline, tratamos as inconsistências e os ruídos encontrados na base de dados bruta da Distribuidora Vetor, preparando os campos para análises agregadas confiáveis.

### Principais Frentes de Atuação:
- **Campos Cadastrais (CNPJ, Telefones e CEP):** Remoção de caracteres especiais (pontos, barras, traços, parênteses e espaços) e do DDI (`+55`/`55`) do início dos telefones para padronizar os dados em formatos limpos.
- **Data de Cadastro:** Conversão de múltiplos formatos misturados (ISO, padrão brasileiro e texto) para o tipo data padrão, tratando falhas de conversão.
- **Normalização Textual de Bairros:** Aplicação de limpeza de texto (minúsculas, remoção de acentos e espaços extras) e estruturação de um dicionário "De-Para" para unificar grafias divergentes.
- **Tratamento de Nulos:** Identificação e substituição de valores ausentes em campos essenciais como `bairro_informado` e `cep` pelo rótulo `"Não informado"`.

**Resumo da etapa:** Limpeza de caracteres especiais e remoção do DDI (`+55`) nas colunas `telefone`, `cnpj` e `cep` aplicadas diretamente nas colunas principais, tratamento dos valores nulos nas colunas `bairro_informado` e `cep` (substituindo por `"Não informado"`), e conversão da coluna `data_cadastro` para o formato `datetime` na cópia de trabalho (`df_analise`).

In [104]:
display(df_clientes['cnpj']) # coluna cnpj do dataframe


# remove o '+55' ou '55' apenas se estiver no início da linha
df_clientes['telefone'] = df_clientes['telefone'].astype(str).str.replace(r'^\+?55', '', regex=True)

# remove caracteres não numéricos da coluna cnpj e cria uma nova coluna
df_clientes['telefone'] = df_clientes['telefone'].str.replace(r'\D', '', regex=True) 
df_clientes['cnpj'] = df_clientes['cnpj'].str.replace(r'\D', '', regex=True)  
df_clientes['cep'] = df_clientes['cep'].str.replace(r'\D', '', regex=True) 

df_analise = df_clientes.copy() # cria uma cópia do dataframe df_clientes para df_analise para realizar as transformações

# substitui os valores nulos da coluna bairro_informado por 'Não informado' usando o método fillna()
df_analise['cep'] = df_analise['cep'].fillna('Não informado') 
df_analise['bairro_informado'] = df_analise['bairro_informado'].fillna('Não informado') 

# converte datas de formatos mistos (meses em texto como dec/aug, barras e ISO) para o padrão ISO (AAAA-MM-DD) do SQLite e Django
df_analise['data_cadastro'] = pd.to_datetime(df_analise['data_cadastro'], format='mixed', dayfirst=True, errors='coerce')

display(df_analise['data_cadastro'].unique()) # exibe os valores únicos da data_cadastro para confirmar que ficaram no formato correto

display(df_analise.head(10)) # ou simplesmente escrever o nome da variável na última linha da célula: df_analise

df_analise.info() # mostra informações do dataframe df_analise para confirmar que o valor nulo foi tratado



0     84.959.310/3413-16
1     28.327.648/3503-05
2     42.388.496/9653-28
3     69.784.801/8451-46
4     89.325.288/0957-01
5     82.278.248/9638-34
6     83.930.103/1051-83
7     31.165.667/0106-51
8     10.801.326/7736-02
9     34.309.805/0097-88
10    93.990.916/9985-43
11    79.911.838/4251-35
12    24.118.244/9353-48
13        27868011280598
14    69.232.260/2563-42
15        36541458685014
16    93.406.088/3561-59
17    82.366.299/4680-44
18        48951343320037
19    20.163.287/0831-72
20    77.434.873/4714-34
21        66587603669096
22        34670656272980
23    37.556.464/1708-05
24        27193745299124
25        49190586518506
26    87.769.453/1473-79
27    49.480.831/3678-37
28    85.685.574/4431-35
29    34.352.408/2400-84
30    04.711.671/9022-94
31    74.964.990/9133-41
32    03.447.134/9361-83
33    17.464.887/7190-65
34    02.787.429/6717-56
35    07.154.516/8087-60
36    10.932.480/8613-17
37    78.263.982/1465-84
38    58.867.533/9636-05
39    71.870.262/1745-96


<DatetimeArray>
['2022-06-21 00:00:00', '2023-07-10 00:00:00', '2022-02-28 00:00:00',
 '2023-12-24 00:00:00', '2022-01-08 00:00:00', '2021-08-17 00:00:00',
 '2025-04-16 00:00:00', '2022-08-18 00:00:00', '2023-09-08 00:00:00',
 '2021-03-03 00:00:00', '2022-09-07 00:00:00', '2023-09-15 00:00:00',
 '2024-03-11 00:00:00', '2023-05-02 00:00:00', '2022-08-28 00:00:00',
 '2024-04-10 00:00:00', '2023-02-28 00:00:00', '2024-08-15 00:00:00',
 '2025-07-15 00:00:00', '2025-08-25 00:00:00', '2022-02-18 00:00:00',
 '2024-10-07 00:00:00', '2022-07-21 00:00:00', '2022-01-25 00:00:00',
 '2024-10-10 00:00:00', '2024-12-11 00:00:00', '2024-11-01 00:00:00',
 '2022-05-03 00:00:00', '2022-12-25 00:00:00', '2025-01-28 00:00:00',
 '2025-07-18 00:00:00', '2022-01-03 00:00:00', '2025-04-03 00:00:00',
 '2022-02-22 00:00:00', '2024-08-20 00:00:00', '2023-12-16 00:00:00',
 '2025-10-27 00:00:00', '2022-11-22 00:00:00', '2025-11-28 00:00:00',
 '2022-12-07 00:00:00', '2024-02-06 00:00:00', '2021-11-19 00:00:00',
 '20

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84959310341316,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28327648350305,61953862084,contato@comercialsão.com.br,Sul (Aguas Claras),72140010,Brasília,DF,2023-07-10,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42388496965328,61924891771,contato@armarinhossi.com.br,Taguatinga - DF,72140010,Brasília,DF,2022-02-28,Camila Torres
3,CLI0004,Empório Vale Verde,69784801845146,61984331053,contato@empóriovalev.com.br,aguas claras,71929000,Brasília,DF,2023-12-24,Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89325288095701,61951273847,contato@merceariadoi.com.br,Taguatinga/DF,00000000,Brasília,DF,2022-01-08,Diego Martins
5,CLI0006,Casa das Tintas Ipê,82278248963834,61971188177,contato@casadastinta.com.br,Águas Claras,Não informado,Brasília,DF,2021-08-17,Camila Torres
6,CLI0007,Distribuidora Popular Centro-Oeste,83930103105183,61999788677,contato@distribuidor.com.br,TAGUATINGA,Não informado,Brasília,DF,2025-04-16,Camila Torres
7,CLI0008,Bazar Estrela Dourada,31165667010651,61941394116,contato@bazarestrela.com.br,V. Pires,72006400,Brasília,DF,2022-08-18,Juliana Reis
8,CLI0009,Supermercado Nova Aliança,10801326773602,61910357396,contato@supermercado.com.br,Aguas claras,71926750,Brasília,DF,2023-09-08,Juliana Reis
9,CLI0010,Papelaria Horizonte,34309805009788,61919319320,contato@papelariahor.com.br,Águas Claras/DF,71950000,Brasília,DF,2021-03-03,Diego Martins


<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_cliente            47 non-null     str           
 1   razao_social          47 non-null     str           
 2   cnpj                  47 non-null     str           
 3   telefone              47 non-null     str           
 4   email                 47 non-null     str           
 5   bairro_informado      47 non-null     str           
 6   cep                   47 non-null     str           
 7   cidade                47 non-null     str           
 8   uf                    47 non-null     str           
 9   data_cadastro         47 non-null     datetime64[us]
 10  vendedor_responsavel  47 non-null     str           
dtypes: datetime64[us](1), str(10)
memory usage: 4.2 KB


### 🧹 Padronização de Texto (`bairro_informado`)

Nesta etapa, aplicamos transformações de string na coluna `bairro_informado` para garantir a consistência e limpeza dos dados antes do mapeamento:

- **Remoção de Espaços (`.str.strip()`):** Elimina espaços em branco indesejados no início e no final do texto.
- **Padronização de Caixa (`.str.title()`):** Converte a primeira letra de cada palavra em maiúscula e as demais em minúscula (ex: `"aguas claras"` vira `"Aguas Claras"`).
- **Validação Estrutural:** Exibe o DataFrame atualizado e as informações técnicas (`.info()`) para confirmar que as alterações foram aplicadas corretamente.

In [105]:
# remove espaços em branco no início e no final da string
df_analise['bairro_informado'] = df_analise['bairro_informado'].str.strip() 

# converte a primeira letra de cada palavra para maiúscula e as demais para minúscula
df_analise['bairro_informado'] = df_analise['bairro_informado'].str.title() 

display(df_analise) 
display(df_analise.info())

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84959310341316,61970855700,contato@mercadinhobo.com.br,Sul (Águas Claras),71926750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28327648350305,61953862084,contato@comercialsão.com.br,Sul (Aguas Claras),72140010,Brasília,DF,2023-07-10,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42388496965328,61924891771,contato@armarinhossi.com.br,Taguatinga - Df,72140010,Brasília,DF,2022-02-28,Camila Torres
3,CLI0004,Empório Vale Verde,69784801845146,61984331053,contato@empóriovalev.com.br,Aguas Claras,71929000,Brasília,DF,2023-12-24,Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89325288095701,61951273847,contato@merceariadoi.com.br,Taguatinga/Df,00000000,Brasília,DF,2022-01-08,Diego Martins
5,CLI0006,Casa das Tintas Ipê,82278248963834,61971188177,contato@casadastinta.com.br,Águas Claras,Não informado,Brasília,DF,2021-08-17,Camila Torres
6,CLI0007,Distribuidora Popular Centro-Oeste,83930103105183,61999788677,contato@distribuidor.com.br,Taguatinga,Não informado,Brasília,DF,2025-04-16,Camila Torres
7,CLI0008,Bazar Estrela Dourada,31165667010651,61941394116,contato@bazarestrela.com.br,V. Pires,72006400,Brasília,DF,2022-08-18,Juliana Reis
8,CLI0009,Supermercado Nova Aliança,10801326773602,61910357396,contato@supermercado.com.br,Aguas Claras,71926750,Brasília,DF,2023-09-08,Juliana Reis
9,CLI0010,Papelaria Horizonte,34309805009788,61919319320,contato@papelariahor.com.br,Águas Claras/Df,71950000,Brasília,DF,2021-03-03,Diego Martins


<class 'pandas.DataFrame'>
RangeIndex: 47 entries, 0 to 46
Data columns (total 11 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_cliente            47 non-null     str           
 1   razao_social          47 non-null     str           
 2   cnpj                  47 non-null     str           
 3   telefone              47 non-null     str           
 4   email                 47 non-null     str           
 5   bairro_informado      47 non-null     str           
 6   cep                   47 non-null     str           
 7   cidade                47 non-null     str           
 8   uf                    47 non-null     str           
 9   data_cadastro         47 non-null     datetime64[us]
 10  vendedor_responsavel  47 non-null     str           
dtypes: datetime64[us](1), str(10)
memory usage: 4.2 KB


None

### 🏙️ Padronização Avançada e Mapeamento de Bairros

Nesta etapa, consolidamos a limpeza da coluna `bairro_informado` utilizando um dicionário de correspondências para unificar todas as variações encontradas:

- **Limpeza Prévia:** Conversão dos dados para string, remoção de espaços extras nas pontas (`.str.strip()`) e padronização das iniciais em maiúsculas (`.str.title()`).
- **Dicionário de Mapeamento (`mapeamentoBairros`):** Agrupamento e correção de variações de digitação, ausência de acentos, abreviações (ex: `V. Pires`) e sufixos regionais (como `- Df`, `/Df` e indicações de Norte/Sul) para os nomes oficiais consolidados.
- **Validação Final:** Execução do `.unique()` para conferir se restou alguma inconsistência e exibição das primeiras linhas do DataFrame (`df_analise.head(10)`) com todas as informações atualizadas.

In [106]:
# 1. Limpeza básica: transforma tudo em string, remove espaços extras e padroniza as iniciais
df_analise['bairro_informado'] = df_analise['bairro_informado'].astype(str).str.strip().str.title()

# 2. Seu dicionário de mapeamento (ajustado para o padrão do .title() que deixa o 'f' de Df minúsculo)
mapeamentoBairros = {
    "Aguas Claras": "Águas Claras",
    "Águas Claras": "Águas Claras",
    "Águas Claras/Df": "Águas Claras",
    "Aguas Claras - Df": "Águas Claras",
    "Águas Claras Norte": "Águas Claras",
    "Norte (Aguas Claras)": "Águas Claras",
    "Águas Claras Sul": "Águas Claras",
    "Sul (Águas Claras)": "Águas Claras",
    "Sul (Aguas Claras)": "Águas Claras",
    "Norte (Águas Claras)": "Águas Claras",

    "Taguatinga": "Taguatinga",
    "Taguatinga - Df": "Taguatinga",
    "Taguatinga Sul": "Taguatinga",
    "Taguatinga/Df": "Taguatinga",

    "Vicente Pires": "Vicente Pires",
    "Vicente Pires - Df": "Vicente Pires",
    "Vte Pires": "Vicente Pires",
    "V. Pires": "Vicente Pires",

    "Guará Ii": "Guará II",
    "Guara I": "Guará I",
    "Guara - Df": "Guará I",
}

# 3. Aplica a substituição
df_analise['bairro_informado'] = df_analise['bairro_informado'].replace(mapeamentoBairros) 

# 4. Exibe os valores únicos para você conferir se limpou tudo
display(df_analise['bairro_informado'].unique())

display(df_analise.head(10)) 

<StringArray>
[ 'Águas Claras',    'Taguatinga', 'Vicente Pires',       'Guará I',
      'Guará II', 'Não Informado']
Length: 6, dtype: str

,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel
0,CLI0001,Mercadinho Boa Vista,84959310341316,61970855700,contato@mercadinhobo.com.br,Águas Claras,71926750,Brasília,DF,2022-06-21,Bruno Almeida
1,CLI0002,Comercial São Judas,28327648350305,61953862084,contato@comercialsão.com.br,Águas Claras,72140010,Brasília,DF,2023-07-10,Camila Torres
2,CLI0003,Armarinhos Silva & Cia,42388496965328,61924891771,contato@armarinhossi.com.br,Taguatinga,72140010,Brasília,DF,2022-02-28,Camila Torres
3,CLI0004,Empório Vale Verde,69784801845146,61984331053,contato@empóriovalev.com.br,Águas Claras,71929000,Brasília,DF,2023-12-24,Diego Martins
4,CLI0005,Mercearia Dois Irmãos,89325288095701,61951273847,contato@merceariadoi.com.br,Taguatinga,00000000,Brasília,DF,2022-01-08,Diego Martins
5,CLI0006,Casa das Tintas Ipê,82278248963834,61971188177,contato@casadastinta.com.br,Águas Claras,Não informado,Brasília,DF,2021-08-17,Camila Torres
6,CLI0007,Distribuidora Popular Centro-Oeste,83930103105183,61999788677,contato@distribuidor.com.br,Taguatinga,Não informado,Brasília,DF,2025-04-16,Camila Torres
7,CLI0008,Bazar Estrela Dourada,31165667010651,61941394116,contato@bazarestrela.com.br,Vicente Pires,72006400,Brasília,DF,2022-08-18,Juliana Reis
8,CLI0009,Supermercado Nova Aliança,10801326773602,61910357396,contato@supermercado.com.br,Águas Claras,71926750,Brasília,DF,2023-09-08,Juliana Reis
9,CLI0010,Papelaria Horizonte,34309805009788,61919319320,contato@papelariahor.com.br,Águas Claras,71950000,Brasília,DF,2021-03-03,Diego Martins


### 🌐 Enriquecimento de Dados com a API do ViaCEP

Nesta etapa final, integramos o pipeline à API pública do ViaCEP para enriquecer nossa base com os endereços oficiais dos clientes a partir do CEP tratado:

- **Busca Automatizada:** Consulta dos logradouros (ruas, avenidas, alamedas) de forma automática.
- **Otimização de Requisições:** Uso de um dicionário de *cache* contendo apenas os CEPs únicos da base, evitando requisições repetidas e garantindo rapidez na execução.
- **Tratamento de Exceções:** Validação prévia de CEPs ausentes ou fora do formato padrão para evitar erros e lentidão durante as chamadas de rede.

In [107]:
# crio uma função pra buscar o logradouro direto na api dos correios
def consultar_viacep(cep):
    cep_str = str(cep).strip()
    
    # se o cep for inválido nem perco tempo consultando
    if cep_str == 'Não informado' or len(cep_str) != 8:
        return 'Não informado'
    
    url = f"https://viacep.com.br/ws/{cep_str}/json/"
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            dados = response.json()
            if 'erro' not in dados:
                return dados.get('logradouro', 'Não informado')
    except requests.exceptions.RequestException:
        pass
        
    return 'Não informado'
    
    

In [109]:

#remove a coluna caso ela exista no DataFrame, evitando erros se a coluna não estiver presente
df_analise = df_analise.drop(columns=['bairro_api'], errors='ignore')

# pego só os ceps unicos pra não repetir requisição a toa
ceps_unicos = df_analise['cep'].unique()

# crio um cache consultando cada cep único
cache_logradouro = {cep: consultar_viacep(cep) for cep in ceps_unicos}

# jogo o logradouro da api de volta na tabela principal usando o cache
df_analise['logradouro_api'] = df_analise['cep'].map(cache_logradouro)

display(df_analise[['cep', 'bairro_informado', 'logradouro_api']].head(10))

display(df_analise.head(16))


,cep,bairro_informado,logradouro_api
0,71926750,Águas Claras,Rua Baracat
1,72140010,Águas Claras,Quadra QNJ 1
2,72140010,Taguatinga,Quadra QNJ 1
3,71929000,Águas Claras,Rua 28
4,00000000,Taguatinga,Não informado
5,Não informado,Águas Claras,Não informado
6,Não informado,Taguatinga,Não informado
7,72006400,Vicente Pires,Rua Rua 6
8,71926750,Águas Claras,Rua Baracat
9,71950000,Águas Claras,Quadra QS 1


,id_cliente,razao_social,cnpj,telefone,email,bairro_informado,cep,cidade,uf,data_cadastro,vendedor_responsavel,logradouro_api
0,CLI0001,Mercadinho Boa Vista,84959310341316,61970855700,contato@mercadinhobo.com.br,Águas Claras,71926750,Brasília,DF,2022-06-21,Bruno Almeida,Rua Baracat
1,CLI0002,Comercial São Judas,28327648350305,61953862084,contato@comercialsão.com.br,Águas Claras,72140010,Brasília,DF,2023-07-10,Camila Torres,Quadra QNJ 1
2,CLI0003,Armarinhos Silva & Cia,42388496965328,61924891771,contato@armarinhossi.com.br,Taguatinga,72140010,Brasília,DF,2022-02-28,Camila Torres,Quadra QNJ 1
3,CLI0004,Empório Vale Verde,69784801845146,61984331053,contato@empóriovalev.com.br,Águas Claras,71929000,Brasília,DF,2023-12-24,Diego Martins,Rua 28
4,CLI0005,Mercearia Dois Irmãos,89325288095701,61951273847,contato@merceariadoi.com.br,Taguatinga,00000000,Brasília,DF,2022-01-08,Diego Martins,Não informado
5,CLI0006,Casa das Tintas Ipê,82278248963834,61971188177,contato@casadastinta.com.br,Águas Claras,Não informado,Brasília,DF,2021-08-17,Camila Torres,Não informado
6,CLI0007,Distribuidora Popular Centro-Oeste,83930103105183,61999788677,contato@distribuidor.com.br,Taguatinga,Não informado,Brasília,DF,2025-04-16,Camila Torres,Não informado
7,CLI0008,Bazar Estrela Dourada,31165667010651,61941394116,contato@bazarestrela.com.br,Vicente Pires,72006400,Brasília,DF,2022-08-18,Juliana Reis,Rua Rua 6
8,CLI0009,Supermercado Nova Aliança,10801326773602,61910357396,contato@supermercado.com.br,Águas Claras,71926750,Brasília,DF,2023-09-08,Juliana Reis,Rua Baracat
9,CLI0010,Papelaria Horizonte,34309805009788,61919319320,contato@papelariahor.com.br,Águas Claras,71950000,Brasília,DF,2021-03-03,Diego Martins,Quadra QS 1


### 🔍 Normalização Automática de Bairros (Fuzzy Matching)

Nesta etapa, automatizamos a padronização das Regiões Administrativas do Distrito Federal:

- **Comparação Aproximada (`difflib`):** Utiliza algoritmos de *fuzzy matching* para detetar e corrigir automaticamente erros de digitação e variações de escrita.
- **Lista Oficial de RAs:** Valida os dados informados contra a lista oficial completa com todas as 35 Regiões Administrativas do DF.
- **Tratamento Seguro:** Caso o texto seja totalmente desconhecido ou fora do padrão, o pipeline preserva a formatação original sem forçar correspondências incorretas.

In [ ]:

# Lista oficial com todas as Regiões Administrativas do Distrito Federal
bairros_oficiais = [
    "Plano Piloto", "Gama", "Taguatinga", "Brazlândia", "Sobradinho", 
    "Planaltina", "Paranoá", "Núcleo Bandeirante", "Ceilândia", "Guará", 
    "Cruzeiro", "Samambaia", "Santa Maria", "São Sebastião", "Recanto das Emas", 
    "Lago Sul", "Riacho Fundo", "Lago Norte", "Candangolândia", "Águas Claras", 
    "Riacho Fundo II", "Sudoeste/Octogonal", "Varjão", "Park Way", "SCIA", 
    "Sobradinho II", "Jardim Botânico", "Itapoã", "SIA", "Vicente Pires", 
    "Fercal", "Sol Nascente/Pôr do Sol", "Arniqueira", "Arapoanga", "Água Quente"
]
